# DAY 2: QLoRA & 4-Bit NormalFloat Quantization

## 🎯 Learning Objectives
By the end of this day, you will be able to explain and demonstrate:
1. **Quantization Principles**: How floating point weights (FP32/FP16) are mapped to discrete 4-bit representations.
2. **NormalFloat4 (NF4)**: Why information-theoretically optimal quantile binning outperforms standard uniform INT4 for Gaussian-distributed neural network weights.
3. **Double Quantization (DQ)**: Quantizing the quantization constants (saving ~0.37 bits/parameter).
4. **Paged Optimizers**: Mitigating memory spikes via CUDA Unified Memory paging to CPU RAM.
5. **Memory Breakdown**: VRAM distribution across weights, activations, gradients, and optimizer states.
6. **LoRA on 4-bit Weights**: Backpropagating gradients through frozen 4-bit quantized base weights into FP16/BF16 adapter matrices.

## 📐 The Three Innovations of QLoRA (Dettmers et al., 2023)

```
Pretrained Weights (FP16)
        ↓  (Quantized to 4-bit NF4 with Double Quantization)
Frozen 4-Bit Base Weights (in VRAM: ~1.1 GB for 1.5B model)
        ↓  (Attached)
Trainable LoRA Adapters (in FP16/BF16: ~18M params)
        ↓  (Forward: Dequantize on-the-fly to compute dtype)
Loss & Backward Pass → Updates only FP16 LoRA Adapters using Paged 8-bit AdamW
```

### 1. 4-bit NormalFloat (NF4)
Standard INT4 uses uniform intervals. However, pretrained LLM weights follow a zero-mean normal distribution $\mathcal{N}(0, \sigma^2)$. NF4 defines bin boundaries such that each of the $2^4 = 16$ bins contains an equal number of expected parameters (equal-area quantiles), minimizing information loss.

### 2. Double Quantization (DQ)
In block-wise quantization (e.g. block size 64), each block has a FP32 scale factor constant $c_1$ (costing $32/64 = 0.5$ bits/param). Double Quantization quantizes $c_1$ into 8-bit FP8 with block size 256, reducing scale constant overhead from 0.5 bits to $8/64 + 32/(64 \times 256) \approx 0.127$ bits/param (saving $0.37$ bits/param).

### 3. Paged Optimizers
Uses CUDA Unified Memory to automatically page page-locked optimizer memory between GPU and CPU during memory spikes (e.g., long sequences in mini-batches), preventing Out-Of-Memory (OOM) fatal errors.

In [ ]:
# Install dependencies if in Colab
# !pip install -q transformers peft accelerate datasets bitsandbytes

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Check CUDA VRAM
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU: {gpu_name} ({total_vram:.2f} GB VRAM)")
else:
    print("Running in CPU mode. (Note: 4-bit bitsandbytes quantization requires CUDA GPU).")

### Step 1: Configure BitsAndBytes 4-Bit NF4 Quantization

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat 4-bit
    bnb_4bit_use_double_quant=True,         # Double quantization (quantize scale constants)
    bnb_4bit_compute_dtype=compute_dtype    # Dequantize to float16/bfloat16 during matrix math
)

print(f"BitsAndBytes Configured: NF4=True, DoubleQuant=True, ComputeDtype={compute_dtype}")

### Step 2: Load 4-Bit Base Model and Inspect VRAM

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    vram_before = torch.cuda.memory_allocated(0) / (1024**2)

    print(f"Loading {MODEL_ID} in 4-bit NF4...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    
    qlora_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=compute_dtype
    )

    vram_after = torch.cuda.memory_allocated(0) / (1024**2)
    print("=" * 60)
    print(f"✅ 4-Bit Base Model Loaded into GPU Memory")
    print(f"VRAM Consumed by 4-bit Base Model: {vram_after - vram_before:.2f} MB (~{(vram_after - vram_before)/1024:.2f} GB)")
    print(f"(Compare to ~3.1 GB required for unquantized FP16 weights!)")
    print("=" * 60)
else:
    print("CPU mode: Loading in standard float32 for pipeline testing...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    qlora_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

### Step 3: Prepare Model for k-bit Training & Attach LoRA

In [ ]:
# Cast layer norms to FP32 and freeze base parameters with gradient checkpointing
if torch.cuda.is_available():
    qlora_model = prepare_model_for_kbit_training(qlora_model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

peft_qlora_model = get_peft_model(qlora_model, lora_config)
peft_qlora_model.print_trainable_parameters()

### Step 4: Verify Inference & Output Generation

In [ ]:
test_prompt = "Extract order into JSON: Sarah bought 5 monitors for $1500 and requested delivery on Monday."
messages = [
    {"role": "system", "content": 'Extract order into JSON: {"customer": str, "quantity": int, "product": str, "amount": float, "delivery_day": str}'},
    {"role": "user", "content": test_prompt}
]

formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted, return_tensors="pt").to(peft_qlora_model.device)

with torch.no_grad():
    outputs = peft_qlora_model.generate(**inputs, max_new_tokens=80, do_sample=False)

print("4-Bit Quantized Base Output:")
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

---
## 🎓 Day 2 Interview Preparation & Concept Mastery

### 🧠 Concepts You Must Know
1. **Uniform Quantization vs NormalFloat (NF4)**: Uniform quantization wastes representation capacity on tails. NF4 assigns equal probability mass to every quantization interval.
2. **Double Quantization**: Compressing the scale constants of the first quantization round, yielding ~0.37 bits/parameter savings.
3. **Compute Dtype vs Storage Dtype**: In QLoRA, base weights are *stored* in 4-bit NF4, but *dequantized on-the-fly* into 16-bit (FP16 or BF16) during forward/backward tensor computations.
4. **Gradient Checkpointing**: Discarding intermediate activation tensors during the forward pass and recomputing them during the backward pass, trading ~20-30% compute time for massive (up to 70%) VRAM savings.
5. **Paged 8-bit AdamW**: Using page-locked host memory via CUDA Unified Memory drivers to handle transient activation and optimizer memory surges without OOM crashes.

### 💼 5 Interview Questions
1. **Q1 (Beginner)**: *What is the difference between LoRA and QLoRA?*
   - **Answer**: Standard LoRA loads the base model in 16-bit or 32-bit precision and trains low-rank adapter matrices. QLoRA quantizes the base model weights to 4-bit (NF4) with double quantization, and uses paged optimizers, reducing base model memory by ~70% while achieving matching task performance.
2. **Q2 (Intermediate)**: *Why can we not simply train the 4-bit quantized weights directly?*
   - **Answer**: Discrete 4-bit values cannot receive small continuous gradient updates (loss gradients $\nabla W$ are tiny floating point numbers like $10^{-5}$ that round to zero on integer scales). QLoRA freezes the 4-bit base weights and only updates high-precision (FP16/BF16) adapter parameters.
3. **Q3 (Intermediate)**: *What is the role of `prepare_model_for_kbit_training` in PEFT?*
   - **Answer**: It casts LayerNorm and embedding layers to FP32 for numerical stability, enables gradient computation on model inputs, and sets up gradient checkpointing hooks.
4. **Q4 (Advanced)**: *Why does BF16 work better than FP16 for compute dtype on Ampere/Hopper GPUs?*
   - **Answer**: BF16 (Bfloat16) has the same 8-bit dynamic dynamic exponent range as FP32 (preventing underflow/overflow spikes during gradient backpropagation), whereas FP16 only has 5 exponent bits and requires careful loss scaling.
5. **Q5 (Advanced)**: *How does Gradient Accumulation enable training with larger effective batch sizes on a 16GB T4?*
   - **Answer**: By computing gradients across micro-batches (e.g. size 2) without stepping the optimizer immediately, summing gradients over $N$ steps (`gradient_accumulation_steps = 4`), and calling `optimizer.step()` only every 4th step, achieving an effective batch size of $2 \times 4 = 8$ with the memory footprint of batch size 2.

### 🛠️ 3 Practical Questions on Your Code
1. *In your code, how did you verify that base weights were loaded in 4-bit rather than 16-bit?*
2. *What parameter in `BitsAndBytesConfig` enables Double Quantization?*
3. *How much VRAM did the 1.5B model occupy before vs. after applying NF4 quantization?*

### 🏆 Day 2 Hands-On Challenge
> **Challenge**: In `BitsAndBytesConfig`, toggle `bnb_4bit_use_double_quant=False` and measure the exact difference in allocated VRAM. Calculate whether the theoretical 0.37 bits/parameter savings matches your measured difference.